# Lab 6 — Transform, model & tune (Exam section 3 · 22%)

**How this lab works**
- Run cells top to bottom (`Shift+Enter`). After a kernel restart, run again from the top.
- Cells marked `# TODO` contain `<FILL-IN>`: replace it with your code, then run the `check(...)` cell.
- Stuck? Ask Claude in the terminal: *"hint for 1c"*. Solutions are in `notebooks/solutions/` (try first!).
- 🖥 **Local** means open-source Spark + Delta on your Mac. ☁ **Databricks** shows the real syntax for the exam.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # lets the notebook import labutils
from labutils import get_spark, path, reset_lab, show, checkpoint, table_path, landing
from labutils.check import check
from pyspark.sql import functions as F
spark = get_spark()
reset_lab("lab6")
spark.sql("""CREATE TABLE lab6.customers_bronze AS SELECT * FROM VALUES
  ('1', '  Ana Silva ', 'ana@x.com',  '2024-01-05', 'us', 'vip'),
  ('2', 'Ben Okoro',    NULL,         '2024-02-11', 'eu', NULL),
  (NULL,'Ghost Row',    'g@x.com',    '2024-03-01', 'us', 'bad id'),
  ('3', 'Chen Li',      'chen@x.com', '2024-03-20', 'EU', NULL),
  ('4', 'Dina Haddad',  NULL,         '2024-04-02', 'us', 'new')
  AS t(customer_id, full_name, email, signup_date, country, raw_note)""")
spark.sql("""CREATE TABLE lab6.orders AS SELECT * FROM VALUES
  (101, 1, 'US', 120.0), (102, 1, 'US', 80.0), (103, 2, 'EU', 45.5), (104, 3, 'EU', 300.0),
  (105, 9, 'US', 10.0),  (105, 9, 'US', 10.0), (106, 4, 'US', 99.9), (107, 3, 'US', 15.0)
  AS t(order_id, customer_id, country, amount)""")

## 6a · Clean bronze into silver

In [ ]:
bronze = spark.table("lab6.customers_bronze")
silver = (bronze
    .na.drop(subset=["customer_id"])
    .na.fill({"email": "unknown"})
    .withColumn("customer_id", F.col("customer_id").cast("int"))
    .withColumn("full_name", F.trim("full_name"))
    .withColumn("first_name", F.split("full_name", " ").getItem(0))
    .withColumn("last_name", F.split("full_name", " ").getItem(1))
    .withColumn("country", F.upper("country"))
    .withColumn("signup_date", F.to_date("signup_date"))
    .withColumnRenamed("signup_date", "signed_up")
    .drop("raw_note"))
silver.write.mode("overwrite").saveAsTable("lab6.customers_silver")
show(spark.table("lab6.customers_silver"))

In [ ]:
check("6a")

## 6b · Joins and unions

In [ ]:
from labutils.submit import submit
orders = spark.table("lab6.orders")
customers = spark.table("lab6.customers_silver")
inner_df = orders.join(customers.select("customer_id", "first_name"), "customer_id", "inner")
left_df  = orders.join(customers.select("customer_id", "first_name"), "customer_id", "left")
multi_df = orders.join(customers.select("customer_id", "country"), ["customer_id", "country"], "inner")
bcast_df = orders.join(F.broadcast(customers), "customer_id", "left")
sizes  = spark.createDataFrame([("S",), ("M",), ("L",)], ["size"])
colors = spark.createDataFrame([("red",), ("blue",)], ["color"])
cross_df = sizes.crossJoin(colors)
eu = spark.createDataFrame([(1, "EU"), (2, "EU")], ["id", "region"])
us = spark.createDataFrame([("US", 3), ("EU", 2)], ["region", "id"])
union_all = eu.unionByName(us)
union_distinct = union_all.distinct()
for name, df in [("inner", inner_df), ("left", left_df), ("multi-key", multi_df), ("cross", cross_df),
                 ("union all", union_all), ("union distinct", union_distinct)]:
    print(f"{name:<15} rows = {df.count()}")
submit("6b", {"inner": inner_df.count(), "left": left_df.count(), "multi": multi_df.count(),
              "cross": cross_df.count(), "union_all": union_all.count(),
              "union_distinct": union_distinct.count(),
              "bcast_plan": bcast_df._jdf.queryExecution().executedPlan().toString()})

In [ ]:
check("6b")

## 6c · De-duplicate and aggregate

In [ ]:
orders = spark.table("lab6.orders")
clean = orders.dropDuplicates(["order_id"])
stats = (clean.groupBy("country")
    .agg(F.count("*").alias("n_orders"),
         F.approx_count_distinct("customer_id").alias("approx_customers"),
         F.avg("amount").alias("avg_amount")))
stats.write.mode("overwrite").saveAsTable("lab6.country_stats")
show(stats)
show(clean.select("amount").summary())

In [ ]:
check("6c")

## 6d · Basic Spark tuning

In [ ]:
from labutils.submit import submit
spark.conf.set("spark.sql.adaptive.enabled", "false")
big = spark.range(100000).withColumn("k", F.col("id") % 100)
small = spark.range(100).withColumnRenamed("id", "k")
plan = lambda df: df._jdf.queryExecution().executedPlan().toString()

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
plan_off = plan(big.join(small, "k"))
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))
plan_on = plan(big.join(small, "k"))
spark.conf.set("spark.sql.shuffle.partitions", "8")
parts = big.groupBy("k").count().rdd.getNumPartitions()

print("threshold -1  ->", "SortMergeJoin" if "SortMergeJoin" in plan_off else plan_off[:80])
print("threshold 10MB->", "BroadcastHashJoin" if "BroadcastHashJoin" in plan_on else plan_on[:80])
print("partitions after groupBy:", parts)
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "2")
submit("6d", {"off": plan_off, "on": plan_on, "parts": parts})

In [ ]:
check("6d")